In [1]:
%pip install -r ../requirements.txt

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### 0. Imports

In [2]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV
from sklearn.cluster import KMeans
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, f1_score, classification_report, adjusted_rand_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist

try:
    import umap
    UMAP_OK = True
except Exception:
    UMAP_OK = False

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_OK = True
except Exception:
    TORCH_OK = False

SEED = 42
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

if TORCH_OK:
    torch.manual_seed(SEED)
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if DEVICE.type == 'cuda':
        torch.cuda.manual_seed_all(SEED)
else:
    DEVICE = None

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 140)
warnings.filterwarnings('ignore')

print('UMAP available:', UMAP_OK)
print('PyTorch available:', TORCH_OK)
if TORCH_OK:
    print('Torch device:', DEVICE,
          '(cuda)' if DEVICE.type == 'cuda' else '(cpu)',
          'GPU:', torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'n/a')

UMAP available: True
PyTorch available: True
Torch device: cpu (cpu) GPU: n/a


### 1. Paths

In [3]:
DATA_ROOT = Path(r"D:\Studi\Bocconi\AILab\ai-lab-gene-expression\data")

SMART = DATA_ROOT / 'SmartSeq'
DROP = DATA_ROOT / 'DropSeq'
OUT_DIR = Path('../outputs')
OUT_DIR.mkdir(exist_ok=True)

assert SMART.exists() and DROP.exists(), f'Data folders not found, check DATA_ROOT'

def load_expr(path):
    df = pd.read_csv(path, sep=r'\s+', engine='python', index_col=0)
    df.index = df.index.astype(str).str.replace('"', '', regex=False)
    df.columns = df.columns.astype(str).str.replace('"', '', regex=False)
    return df

print(f'OK, using DATA_ROOT = {DATA_ROOT}')

OK, using DATA_ROOT = D:\Studi\Bocconi\AILab\ai-lab-gene-expression\data


In [4]:
matrices = {}

matrices[('SmartSeq', 'MCF7', 'unfiltered')]  = load_expr(SMART / 'MCF7_SmartS_Unfiltered_Data.txt')
matrices[('SmartSeq', 'MCF7', 'filtered')]    = load_expr(SMART / 'MCF7_SmartS_Filtered_Data.txt')
matrices[('SmartSeq', 'MCF7', 'normalised')]  = load_expr(SMART / 'MCF7_SmartS_Filtered_Normalised_3000_Data_train.txt')
matrices[('SmartSeq', 'HCC1806', 'unfiltered')] = load_expr(SMART / 'HCC1806_SmartS_Unfiltered_Data.txt')
matrices[('SmartSeq', 'HCC1806', 'filtered')]   = load_expr(SMART / 'HCC1806_SmartS_Filtered_Data.txt')
matrices[('SmartSeq', 'HCC1806', 'normalised')] = load_expr(SMART / 'HCC1806_SmartS_Filtered_Normalised_3000_Data_train.txt')

matrices[('DropSeq', 'MCF7', 'normalised')]    = load_expr(DROP / 'MCF7_Filtered_Normalised_3000_Data_train.txt')
matrices[('DropSeq', 'HCC1806', 'normalised')] = load_expr(DROP / 'HCC1806_Filtered_Normalised_3000_Data_train.txt')

summary_rows = []
for (tech, line, stage), df in matrices.items():
    summary_rows.append({'technology': tech, 'cell_line': line, 'stage': stage,
                         'n_genes': df.shape[0], 'n_cells': df.shape[1]})
summary_df = pd.DataFrame(summary_rows).sort_values(['technology', 'cell_line', 'stage']).reset_index(drop=True)
summary_df

,technology,cell_line,stage,n_genes,n_cells
0,DropSeq,HCC1806,normalised,3000,14682
1,DropSeq,MCF7,normalised,3000,21626
2,SmartSeq,HCC1806,filtered,19503,227
3,SmartSeq,HCC1806,normalised,3000,182
4,SmartSeq,HCC1806,unfiltered,23396,243
5,SmartSeq,MCF7,filtered,18945,313
6,SmartSeq,MCF7,normalised,3000,250
7,SmartSeq,MCF7,unfiltered,22934,383


In [5]:
def extract_label(cell_name: str):
    s = cell_name.lower()
    if '_hypoxia' in s or '_hypo_' in s or s.endswith('_hypo'):
        return 'Hypo'
    if '_normoxia' in s or '_norm_' in s or s.endswith('_norm'):
        return 'Norm'
    return None

meta_rows = []
for (tech, line, stage), df in matrices.items():
    for cell in df.columns:
        meta_rows.append({
            'cell_id': cell,
            'technology': tech,
            'cell_line': line,
            'stage': stage,
            'condition': extract_label(cell),
        })
meta = pd.DataFrame(meta_rows)

labels_by_cell = meta.drop_duplicates('cell_id').set_index('cell_id')['condition']

meta_smart_mcf7 = pd.read_csv(SMART / 'MCF7_SmartS_MetaData.tsv', sep='\t')
meta_smart_hcc  = pd.read_csv(SMART / 'HCC1806_SmartS_MetaData.tsv', sep='\t')
for m in (meta_smart_mcf7, meta_smart_hcc):
    m['Filename'] = m['Filename'].astype(str).str.replace('"', '', regex=False)

print('Cells per (technology, cell_line, condition), counted on the normalised stage only:')
counts = (meta[meta['stage'] == 'normalised']
          .groupby(['technology', 'cell_line', 'condition'])
          .size().unstack(fill_value=0))
counts

Cells per (technology, cell_line, condition), counted on the normalised stage only:


condition             Hypo   Norm
technology cell_line             
DropSeq    HCC1806    8899   5783
           MCF7       8921  12705
SmartSeq   HCC1806      97     85
           MCF7        124    126

In [6]:
def make_Xy(tech, line):
    df = matrices[(tech, line, 'normalised')]
    y = labels_by_cell.reindex(df.columns)
    mask = y.isin(['Hypo', 'Norm']).values
    X = df.T.values[mask]
    y = (y[mask] == 'Hypo').astype(int).values
    genes = df.index.values
    return X, y, genes

SUBSETS = [('SmartSeq', 'MCF7'), ('SmartSeq', 'HCC1806'), ('DropSeq', 'MCF7'), ('DropSeq', 'HCC1806')]
Xys = {s: make_Xy(*s) for s in SUBSETS}
for s, (X, y, _) in Xys.items():
    print(f'{s}: X={X.shape}, class balance={np.bincount(y)}')

('SmartSeq', 'MCF7'): X=(250, 3000), class balance=[126 124]
('SmartSeq', 'HCC1806'): X=(182, 3000), class balance=[85 97]
('DropSeq', 'MCF7'): X=(21626, 3000), class balance=[12705  8921]
('DropSeq', 'HCC1806'): X=(14682, 3000), class balance=[5783 8899]


In [7]:
def k_means_pipiline():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("kmeans", KMeans(
            n_clusters=2,
            random_state=SEED,
            n_init=20
        )),
    ])

def align_clusters(cluster_labels, y_true):
    majority_of_0 = np.bincount(y_true[cluster_labels == 0].astype(np.int64)).argmax()
    aligned = np.where(cluster_labels == 0, majority_of_0, 1 - majority_of_0)
    if accuracy_score(y_true, aligned) < 0.5:
        aligned = 1 - aligned
    return aligned

def k_means(X, y):
    pipe = k_means_pipiline()
    clusters = pipe.fit_predict(X)
    ari = adjusted_rand_score(y, clusters)
    y_pred = align_clusters(clusters, y)
    return ari, y_pred, y

for tech in ('SmartSeq', 'DropSeq'):
    for line in ('MCF7', 'HCC1806'):
        X, y, _ = Xys[(tech, line)]
        ari, _, _ = k_means(X, y)
        print(f'{tech:8s} {line:8s}  ARI = {ari:.3f}')

SmartSeq MCF7      ARI = 0.952
SmartSeq HCC1806   ARI = -0.005
DropSeq  MCF7      ARI = 0.503
DropSeq  HCC1806   ARI = 0.065


In [8]:
def cv_score(clf, X, y, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    scores = cross_validate(
        Pipeline([('sc', StandardScaler()), ('clf', clf)]),
        X, y, cv=skf,
        scoring=['roc_auc', 'accuracy', "balanced_accuracy", 'f1'],
        return_train_score=False,
    )
    return {k.replace('test_', ''): float(np.mean(v)) for k, v in scores.items() if k.startswith('test_')}


In [9]:
def make_knn(tech, k, n_comp=50):
    if tech == "SmartSeq":
        return Pipeline([
            ("sc", StandardScaler()),
            ("pca", PCA(n_components=n_comp, random_state=SEED)),
            ("knn", KNeighborsClassifier(n_neighbors=k, metric="euclidean"))
        ])
    elif tech == "DropSeq":
        return Pipeline([
            ("sc", StandardScaler()),
            ("knn", KNeighborsClassifier(n_neighbors=k, metric="cosine"))
        ])

In [10]:
def best_k(X, y, tech=None):
    k_grid = range(1, 52, 2) if tech == "DropSeq" else range(1, 32, 2)
        
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    search = GridSearchCV(
        Pipeline([("sc", StandardScaler()),
                  ("knn", KNeighborsClassifier(metric="cosine"))]),
        param_grid={"knn__n_neighbors": list(k_grid)},
        cv=inner_cv, scoring="roc_auc", n_jobs=-1
    )
    search.fit(X, y)
    return search.best_params_["knn__n_neighbors"]

In [11]:
for (tech, line), (X, y, g) in Xys.items():
    if tech == "DropSeq" and len(X) > 2000:
        idx = rng.choice(len(X), size=2000, replace=False)
        X, y = X[idx], y[idx]
    print(f'{tech} {line} - {best_k(X, y, tech=tech)}')

SmartSeq MCF7 - 7
SmartSeq HCC1806 - 31
DropSeq MCF7 - 51
DropSeq HCC1806 - 51


In [12]:
in_dist = {}

print("IN-DISTRIBUTION (5-fold CV)")
print(f"{'Subset':25s}  {'Model':4s}  {'AUC':>6}  {'Acc':>6}  {'BalAcc':>7}  {'F1':>6}")
print("-" * 62)

for (tech, line), (X, y, g) in Xys.items():
    if tech == "DropSeq" and len(X) > 2000:
        idx = rng.choice(len(X), size=2000, replace=False)
        X, y = X[idx], y[idx]
    k = best_k(X, y)
    model_name = "KNN"
    clf = make_knn(tech, k)
    
    s = cv_score(clf, X, y)
    in_dist[(tech, line, model_name)] = s | {"k": k if model_name == "KNN" else None}
    print(
        f"{tech+' '+line:25s}  {model_name:4s}  "
        f"{s['roc_auc']:.3f}  {s['accuracy']:.3f}  "
        f"{s['balanced_accuracy']:.3f}  {s['f1']:.3f}"
        + (f"  [k={k}]" if model_name == "KNN" else "")
    )

IN-DISTRIBUTION (5-fold CV)
Subset                     Model     AUC     Acc   BalAcc      F1
--------------------------------------------------------------
SmartSeq MCF7              KNN   1.000  0.996  0.996  0.996  [k=7]
SmartSeq HCC1806           KNN   0.984  0.879  0.884  0.870  [k=31]
DropSeq MCF7               KNN   0.943  0.890  0.870  0.848  [k=31]
DropSeq HCC1806            KNN   0.872  0.780  0.754  0.829  [k=31]


In [13]:
  
def make_mlp() -> Pipeline:
    return MLPClassifier(
        hidden_layer_sizes=(256, 64),
        activation="relu",
        solver="adam",
        alpha=1e-4,
        batch_size=64,
        learning_rate_init=1e-3,
        max_iter=300,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=15,
        random_state=SEED,
    )

In [ ]:
in_dist = {}
print("IN-DISTRIBUTION (5-fold CV)")
print(f"{'Subset':25s}  {'Model':4s}  {'AUC':>6}  {'Acc':>6}  {'BalAcc':>7}  {'F1':>6}")
print("-" * 62)
for (tech, line), (X, y, g) in Xys.items():
    model_name = "MLP"
    clf = make_mlp()
    s = cv_score(clf, X, y)
    in_dist[(tech, line, model_name)] = s
    print(
        f"{tech+' '+line:25s}  {model_name:4s}  "
        f"{s['roc_auc']:.3f}  {s['accuracy']:.3f}  "
        f"{s['balanced_accuracy']:.3f}  {s['f1']:.3f}"
        + (f"  [k={k}]" if model_name == "KNN" else "")
    )
    

IN-DISTRIBUTION (5-fold CV)
Subset                     Model     AUC     Acc   BalAcc      F1
--------------------------------------------------------------
SmartSeq MCF7              MLP   0.999  0.988  0.988  0.988
SmartSeq HCC1806           MLP   0.918  0.884  0.883  0.892


In [ ]:
# add cross-line and cross-sequence